# Notebook 05 — Panel Regression Analysis

This notebook runs the three fixed-effects panel regression models:

| Model | Hypothesis | Key Test |
|---|---|---|
| **H1** | Heat → Mortality | Coefficient on heatwave days |
| **H2** | Heat × Vulnerability | Interaction: hw_days × RSVI |
| **H3** | 2022 Amplification | Three-way: hw_days × RSVI × D2022 |

All models use entity (region) and time (year) fixed effects with
clustered standard errors at the region level.

**Prerequisites:** Run Notebook 04 (panel dataset must exist).

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import load_config, get_path
from src.utils.io import load_dataframe, save_dataframe

cfg = load_config()
print("Ready.")

In [ ]:
# Load panel dataset
panel_path = get_path(cfg, 'processed_data') / 'panel_dataset.csv'
panel = load_dataframe(panel_path)

print(f"Panel: {panel.shape}")
print(f"Regions: {panel['nuts2_code'].nunique()}, Years: {panel['year'].nunique()}")
print(f"\nAvailable variables for regression:")
regression_vars = ['mortality_rate', 'hw_days', 'summer_tmax_anomaly', 'rsvi',
                   'hw_days_x_rsvi', 'tmax_anomaly_x_rsvi', 'd2022',
                   'hw_days_x_rsvi_x_d2022', 'covid_period']
for v in regression_vars:
    status = '✅' if v in panel.columns and not panel[v].isna().all() else '❌'
    print(f"  {status} {v}")

---
## 1. Set Panel Index

The `linearmodels` package requires a MultiIndex of (entity, time).

In [ ]:
from src.analysis.panel_regression import prepare_panel_index

panel_idx = prepare_panel_index(panel)
print(f"Index: {panel_idx.index.names}")
print(f"Shape: {panel_idx.shape}")
display(panel_idx.head())

---
## 2. Model H1 — Does Heat Cause Excess Mortality?

**Model specification:**
```
mortality_rt = β₁·hw_days + β₂·summer_tmax_anomaly + β₃·covid_period + α_r + γ_t + ε_rt
```

If β₁ > 0 and significant ⟹ more heatwave days = higher mortality.

In [ ]:
from src.analysis.panel_regression import run_model_h1

h1 = run_model_h1(panel_idx)

if h1['results'] is not None:
    print(h1['results'].summary)
else:
    print("Model could not be run — check that mortality_rate column exists.")
    print("\nThis model requires mortality data to be loaded.")

---
## 3. Model H2 — Does Vulnerability Amplify the Heat-Mortality Link?

**Model specification:**
```
mortality_rt = β₁·hw_days + β₂·rsvi + β₃·(hw_days × rsvi)
             + β₄·tmax_anomaly + β₅·(tmax_anomaly × rsvi)
             + β₆·covid_period + α_r + γ_t + ε_rt
```

**Key parameter:** β₃ (hw_days × RSVI interaction)
- If β₃ > 0 and significant ⟹ the heat-mortality effect is *stronger* in more vulnerable regions.

In [ ]:
from src.analysis.panel_regression import run_model_h2

h2 = run_model_h2(panel_idx)

if h2['results'] is not None:
    print(h2['results'].summary)
    
    # Highlight key coefficient
    if 'hw_days_x_rsvi' in h2['results'].params.index:
        coef = h2['results'].params['hw_days_x_rsvi']
        pval = h2['results'].pvalues['hw_days_x_rsvi']
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f"\n🔑 KEY RESULT: hw_days × RSVI = {coef:.4f} (p = {pval:.4f}) {sig}")
        if coef > 0 and pval < 0.05:
            print("   → Vulnerable regions experience MORE heat-related mortality.")
else:
    print("Model could not be run.")

---
## 4. Model H3 — Did 2022 Amplify Inequality?

**Model specification:**
```
mortality_rt = ... + β·(hw_days × rsvi × D2022) + α_r + γ_t + ε_rt
```

**Key parameter:** β on the three-way interaction
- If β > 0 and significant ⟹ the 2022 heatwave *widened* the gap between high- and low-vulnerability regions.

In [ ]:
from src.analysis.panel_regression import run_model_h3

h3 = run_model_h3(panel_idx)

if h3['results'] is not None:
    print(h3['results'].summary)
    
    if 'hw_days_x_rsvi_x_d2022' in h3['results'].params.index:
        coef = h3['results'].params['hw_days_x_rsvi_x_d2022']
        pval = h3['results'].pvalues['hw_days_x_rsvi_x_d2022']
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f"\n🔑 KEY RESULT: hw_days × RSVI × D2022 = {coef:.4f} (p = {pval:.4f}) {sig}")
else:
    print("Model could not be run.")

---
## 5. Model Comparison Table

In [ ]:
from src.analysis.panel_regression import extract_results_table

all_results = [h1, h2, h3]
comparison = extract_results_table(all_results)

if not comparison.empty:
    print("Regression Results Comparison:")
    display(comparison.style.format({
        'coefficient': '{:.4f}',
        'std_error': '{:.4f}',
        't_stat': '{:.2f}',
        'p_value': '{:.4f}',
    }))
    
    # Save
    table_dir = get_path(cfg, 'tables')
    save_dataframe(comparison, table_dir / 'regression_results.csv', index=False)
    print("\n✅ Saved to outputs/tables/regression_results.csv")
else:
    print("No results to compare — models could not be run.")

---
## 6. Model Diagnostics

In [ ]:
from src.analysis.diagnostics import compute_vif, residual_diagnostics

# VIF check for multicollinearity
exog_vars = ['hw_days', 'rsvi', 'hw_days_x_rsvi',
             'summer_tmax_anomaly', 'tmax_anomaly_x_rsvi', 'covid_period']
available = [v for v in exog_vars if v in panel.columns]

if len(available) > 1:
    vif = compute_vif(panel, available)
    print("Variance Inflation Factors:")
    display(vif)
    
    if vif['VIF'].max() > 10:
        print("\n⚠️ VIF > 10 detected — consider removing collinear variables.")
    else:
        print("\n✅ No problematic multicollinearity detected.")

In [ ]:
# Residual diagnostics for H2 model
if h2['results'] is not None:
    fig_dir = get_path(cfg, 'figures')
    diag = residual_diagnostics(h2['results'], output_dir=fig_dir)
    
    print("Residual Diagnostics (H2 model):")
    for k, v in diag.items():
        print(f"  {k}: {v}")
else:
    print("No model results for diagnostics.")

In [ ]:
# Sensitivity: exclude 2020 (COVID year)
from src.analysis.diagnostics import sensitivity_exclude_covid

if 'mortality_rate' in panel_idx.columns:
    h2_no_covid = sensitivity_exclude_covid(panel_idx)
    if h2_no_covid['results'] is not None:
        print("Sensitivity Analysis — Excluding 2020:")
        print(h2_no_covid['results'].summary)
else:
    print("Mortality rate needed for sensitivity analysis.")

---
## 7. Interpretation Helper

Use this cell to compute marginal effects.

In [ ]:
# Marginal effect of heat on mortality at different RSVI levels
if h2['results'] is not None:
    params = h2['results'].params
    beta_hw = params.get('hw_days', 0)
    beta_int = params.get('hw_days_x_rsvi', 0)
    
    print("Marginal effect of 1 additional heatwave day on mortality:")
    print(f"  At RSVI = 0.25 (low vulnerability):  {beta_hw + beta_int * 0.25:.4f}")
    print(f"  At RSVI = 0.50 (median):             {beta_hw + beta_int * 0.50:.4f}")
    print(f"  At RSVI = 0.75 (high vulnerability):  {beta_hw + beta_int * 0.75:.4f}")
    print(f"  At RSVI = 1.00 (maximum):            {beta_hw + beta_int * 1.00:.4f}")
else:
    print("Model results needed for marginal effects.")

print("\n➡️ Next: Notebook 06 — Visualization & Results")